# Part 2: Training a Linear Model with Trainable Basis Functions

#### Welcome to Part 2!

In Part 1 we used nnAudio2's `MelSpectrogram` as a fixed feature extractor. The key insight of nnAudio2 is that the filterbank is a genuine `nn.Module` — its weights can be made **trainable**, so the Mel basis and STFT kernels adapt to the task during backpropagation.

This tutorial demonstrates all four trainability configurations and trains the full model end-to-end on the same 12-class keyword spotting task.

**Trainability options** (`MelSpectrogram(trainable_mel=..., trainable_STFT=...)`):

| Config | `trainable_mel` | `trainable_STFT` |
|--------|:--------------:|:---------------:|
| A — fixed front-end      | `False` | `False` |
| B — trainable Mel only   | `True`  | `False` |
| C — trainable STFT only  | `False` | `True`  |
| D — fully trainable      | `True`  | `True`  |

[Step 1: Imports](#Step-1:-Imports)\
[Step 2: Configuration & device](#Step-2:-Configuration-&-device)\
[Step 3: Dataset & DataLoaders](#Step-3:-Dataset-&-DataLoaders)\
[Step 4: Trainable nnAudio model](#Step-4:-Trainable-nnAudio-model)\
[Step 5: Train](#Step-5:-Train)\
[Conclusion](#Conclusion)

## Step 1: Imports

Same imports as Part 1, plus the `SPEECHCOMMANDS_12C` dataset wrapper defined in the next cell.

In [2]:
import os
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.utils.data import WeightedRandomSampler, DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

from pytorch_lightning import Trainer, LightningModule

import librosa
from nnAudio2.features.mel import MelSpectrogram

/var/folders/xm/qzb5tds96wd1w9ss080zx9q40000gn/T/ipykernel_89431/2918432426.py:14: CitationReminderWarning: ============================================================
nnAudio Citation Reminder

If you like nnAudio, please cite:

K. W. Cheuk, H. Anderson, K. Agres and D. Herremans,
"nnAudio: An on-the-Fly GPU Audio to Spectrogram Conversion
Toolbox Using 1D Convolutional Neural Networks,"
IEEE Access, vol. 8, pp. 161981-162003, 2020,
doi: 10.1109/ACCESS.2020.3019084.

  from nnAudio2.features.mel import MelSpectrogram


## Step 2: Configuration & device

Device is detected automatically — CUDA → MPS (Apple Silicon) → CPU.

In [3]:
if torch.cuda.is_available():
    device, accelerator = 'cuda', 'gpu'
elif torch.backends.mps.is_available():
    device, accelerator = 'mps', 'mps'
else:
    device, accelerator = 'cpu', 'cpu'

print(f"Using device: {device}")

batch_size               = 100
max_epochs               = 1
check_val_every_n_epoch  = 2
num_sanity_val_steps     = 5

data_root        = './'    # dataset downloaded here
download_option  = True    # set False once downloaded

n_mels     = 40
input_dim  = n_mels * 101
output_dim = 12

Using device: mps


## Step 3: Dataset & DataLoaders

Same setup as Part 1: weighted sampler to balance silence/unknown classes, padding to 16 000 samples in `collate_fn`.

In [4]:
## SPEECHCOMMANDS_12C: 10 keywords + silence + unknown
# Replaces the obsolete AudioLoader.Speech.SPEECHCOMMANDS_12C with a torchaudio-based wrapper.

KEYWORDS = ['down', 'go', 'left', 'no', 'off', 'on', 'right', 'stop', 'up', 'yes']
LABEL_MAP = {word: i for i, word in enumerate(KEYWORDS)}
SILENCE_LABEL = 10
UNKNOWN_LABEL  = 11
TARGET_LEN = 16000  # 1 second at 16 kHz

class SPEECHCOMMANDS_12C(Dataset):
    """torchaudio SPEECHCOMMANDS wrapped as a 12-class dataset.

    Classes 0-9: the 10 target keywords
    Class 10   : silence (1-second clips cut from background noise files)
    Class 11   : unknown (all remaining words)
    """
    def __init__(self, root, url='speech_commands_v0.02',
                 folder_in_archive='SpeechCommands', download=False, subset=None):
        self.base = torchaudio.datasets.SPEECHCOMMANDS(
            root=root, url=url, folder_in_archive=folder_in_archive,
            download=download, subset=subset,
        )
        self.silence = []
        noise_dir = os.path.join(root, folder_in_archive, '_background_noise_')
        if os.path.exists(noise_dir):
            for fname in sorted(os.listdir(noise_dir)):
                if fname.endswith('.wav'):
                    wav, sr = torchaudio.load(os.path.join(noise_dir, fname))
                    for start in range(0, wav.shape[1] - TARGET_LEN, TARGET_LEN):
                        self.silence.append(wav[:, start:start + TARGET_LEN])

    @staticmethod
    def _fix_length(wav):
        """Pad or truncate to exactly TARGET_LEN samples."""
        n = wav.shape[1]
        if n < TARGET_LEN:
            wav = torch.nn.functional.pad(wav, (0, TARGET_LEN - n))
        elif n > TARGET_LEN:
            wav = wav[:, :TARGET_LEN]
        return wav

    def __len__(self):
        return len(self.base) + len(self.silence)

    def __getitem__(self, idx):
        if idx < len(self.base):
            waveform, sr, label, speaker_id, utt_num = self.base[idx]
            waveform = self._fix_length(waveform)
            return waveform, sr, LABEL_MAP.get(label, UNKNOWN_LABEL), speaker_id, utt_num
        waveform = self.silence[idx - len(self.base)]
        return waveform, 16000, SILENCE_LABEL, '', 0

In [5]:
trainset = SPEECHCOMMANDS_12C(root=data_root, url='speech_commands_v0.02',
                              folder_in_archive='SpeechCommands',
                              download=download_option, subset='training')
validset = SPEECHCOMMANDS_12C(root=data_root, url='speech_commands_v0.02',
                              folder_in_archive='SpeechCommands',
                              download=download_option, subset='validation')

class_weights  = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4.6, 1/17]
sample_weights = [class_weights[label] for _, _, label, _, _ in trainset]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

def collate_fn(batch):
    waveforms = pad_sequence([b[0].squeeze(0) for b in batch], batch_first=True)
    return {'waveforms': waveforms, 'labels': torch.tensor([b[2] for b in batch])}

trainloader = DataLoader(trainset, batch_size=batch_size, sampler=sampler, collate_fn=collate_fn)
validloader = DataLoader(validset, batch_size=batch_size, collate_fn=collate_fn)

print(f"Train: {len(trainset):,} samples  |  Val: {len(validset):,} samples")

/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


Train: 84,843 samples  |  Val: 9,981 samples


In [6]:
if torch.cuda.is_available():
    device, accelerator = 'cuda', 'gpu'
elif torch.backends.mps.is_available():
    device, accelerator = 'mps', 'mps'
else:
    device, accelerator = 'cpu', 'cpu'

print(f"Using device: {device}")

batch_size              = 100
max_epochs              = 200
check_val_every_n_epoch = 2
num_sanity_val_steps    = 5

data_root       = './'    # dataset downloaded here
download_option = True    # set False once downloaded

n_mels     = 40
input_dim  = n_mels * 101
output_dim = 12

Using device: mps


In [7]:
import os
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.utils.data import WeightedRandomSampler, DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

from pytorch_lightning import Trainer, LightningModule

from nnAudio2.features.mel import MelSpectrogram

In [8]:
KEYWORDS    = ['down', 'go', 'left', 'no', 'off', 'on', 'right', 'stop', 'up', 'yes']
LABEL_MAP   = {word: i for i, word in enumerate(KEYWORDS)}
SILENCE_LABEL = 10
UNKNOWN_LABEL = 11
TARGET_LEN    = 16000  # 1 second at 16 kHz

class SPEECHCOMMANDS_12C(Dataset):
    """torchaudio SPEECHCOMMANDS mapped to 12 integer classes.

    0–9: target keywords  |  10: silence  |  11: unknown
    """
    def __init__(self, root, url='speech_commands_v0.02',
                 folder_in_archive='SpeechCommands', download=False, subset=None):
        self.base = torchaudio.datasets.SPEECHCOMMANDS(
            root=root, url=url, folder_in_archive=folder_in_archive,
            download=download, subset=subset,
        )
        self.silence = []
        noise_dir = os.path.join(root, folder_in_archive, '_background_noise_')
        if os.path.exists(noise_dir):
            for fname in sorted(os.listdir(noise_dir)):
                if fname.endswith('.wav'):
                    wav, sr = torchaudio.load(os.path.join(noise_dir, fname))
                    for start in range(0, wav.shape[1] - TARGET_LEN, TARGET_LEN):
                        self.silence.append(wav[:, start:start + TARGET_LEN])

    @staticmethod
    def _fix_length(wav):
        n = wav.shape[1]
        if n < TARGET_LEN:
            return torch.nn.functional.pad(wav, (0, TARGET_LEN - n))
        return wav[:, :TARGET_LEN]

    def __len__(self):
        return len(self.base) + len(self.silence)

    def __getitem__(self, idx):
        if idx < len(self.base):
            waveform, sr, label, speaker_id, utt_num = self.base[idx]
            return self._fix_length(waveform), sr, LABEL_MAP.get(label, UNKNOWN_LABEL), speaker_id, utt_num
        return self.silence[idx - len(self.base)], 16000, SILENCE_LABEL, '', 0

## Step 4: Trainable nnAudio model

The only change from Part 1 is flipping `trainable_mel` and `trainable_STFT` to `True`. Everything else — architecture, training loop, device handling — stays identical.

When `trainable_mel=True`, `mel_basis` becomes an `nn.Parameter` and receives gradients like any other weight. Mel filterbank values must stay non-negative to remain physically meaningful, so we apply `clamp_` after each optimizer step (projected gradient).

When `trainable_STFT=True`, the sinusoidal STFT kernels (`wsin`, `wcos`) are also parameters and adapt freely.

Try the four configs from the intro table by changing the constructor arguments below.

In [9]:
class TrainableKeywordSpotter(LightningModule):
    """Linear keyword spotter with trainable nnAudio2 Mel and STFT basis functions.

    Change trainable_mel / trainable_STFT to explore the four configurations
    described in the intro table (A–D).
    """

    def __init__(self, trainable_mel=True, trainable_STFT=True):
        super().__init__()
        self.mel = MelSpectrogram(
            sr=16000, n_fft=480, hop_length=160, n_mels=n_mels,
            fmin=0.0, norm=1,
            trainable_mel=trainable_mel,
            trainable_STFT=trainable_STFT,
            verbose=False,
        )
        self.classifier = nn.Linear(n_mels * 101, output_dim)
        self.criterion   = nn.CrossEntropyLoss()

    def forward(self, x):
        spec   = torch.log(self.mel(x) + 1e-10)       # [B, n_mels, T]
        logits = self.classifier(spec.flatten(1))      # [B, 12]
        return logits, spec

    def optimizer_step(self, epoch, batch_idx, optimizer, optimizer_closure=None):
        optimizer.step(closure=optimizer_closure)
        # Project mel filterbank weights back to [0, 1] after each update
        with torch.no_grad():
            torch.clamp_(self.mel.mel_basis, 0, 1)

    def _step(self, batch):
        logits, _ = self(batch['waveforms'])
        loss = self.criterion(logits, batch['labels'])
        acc  = (logits.argmax(-1) == batch['labels']).float().mean()
        return loss, acc

    def training_step(self, batch, batch_idx):
        loss, acc = self._step(batch)
        self.log_dict({'train_loss': loss, 'train_acc': acc},
                      on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, acc = self._step(batch)
        self.log_dict({'val_loss': loss, 'val_acc': acc}, prog_bar=True)

    def configure_optimizers(self):
        # Separate learning rates: smaller for basis functions, standard for the classifier
        mel_params        = [p for n, p in self.named_parameters() if 'mel.' in n]
        classifier_params = [p for n, p in self.named_parameters() if 'mel.' not in n]
        return optim.Adam([
            {'params': classifier_params, 'lr': 1e-3},
            {'params': mel_params,        'lr': 1e-4},
        ])

# Config D — both Mel and STFT trainable (change here to try A/B/C)
model = TrainableKeywordSpotter(trainable_mel=True, trainable_STFT=True)
print(model)

TrainableKeywordSpotter(
  (mel): MelSpectrogram(
    Mel filter banks size = (40, 241), trainable_mel=True
    (stft): STFT(n_fft=480, Fourier Kernel size=(241, 1, 480), iSTFT=False, trainable=True)
  )
  (classifier): Linear(in_features=4040, out_features=12, bias=True)
  (criterion): CrossEntropyLoss()
)


## Step 5: Train

Training for 200 epochs lets the basis functions adapt meaningfully. Watch `train_acc` and `val_acc` — with a fully trainable front-end (config D) you should see a lift over the fixed baseline from Part 1.

The trained weights are saved automatically by Lightning in the `lightning_logs/` folder.

In [ ]:
trainer = Trainer(
    accelerator=accelerator,
    devices=1,
    max_epochs=max_epochs,
    check_val_every_n_epoch=check_val_every_n_epoch,
    num_sanity_val_steps=num_sanity_val_steps,
)
trainer.fit(model, trainloader, validloader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default

  | Name       | Type             | Params | Mode 
--------------------------------------------------------
0 | mel        | Me

Sanity Checking DataLoader 0:  80%|████████  | 4/5 [00:00<00:00, 39.18it/s]

/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


/Users/dorien_herremans/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 1: 100%|██████████| 849/849 [00:15<00:00, 54.56it/s, v_num=3, train_loss=5.680, train_acc=0.231]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 849/849 [00:15<00:00, 54.55it/s, v_num=3, train_loss=3.590, train_acc=0.355, val_loss=3.670, val_acc=0.199]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 5: 100%|██████████| 849/849 [00:16<00:00, 51.36it/s, v_num=3, train_loss=3.250, train_acc=0.392, val_loss=4.570, val_acc=0.175]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 7: 100%|██████████| 849/849 [00:15<00:00, 53.72it/s, v_num=3, train_loss=3.160, train_acc=0.419, val_loss=6.060, val_acc=0.190]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 9: 100%|██████████| 849/849 [00:15<00:00, 55.08it/s, v_num=3, train_loss=3.090, train_acc=0.432, val_loss=3.740, val_acc=0.303]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 11: 100%|██████████| 849/849 [00:16<00:00, 51.29it/s, v_num=3, train_loss=2.910, train_acc=0.455, val_loss=3.960, val_acc=0.2

## Conclusion

By setting `trainable_mel=True` and `trainable_STFT=True`, the Mel filterbank and STFT kernels are updated alongside the classifier during training. The non-negativity constraint on `mel_basis` is enforced via a post-step clamp, keeping the filterbank physically meaningful.

The trained weights are in `lightning_logs/`. In **Part 3** we load them and evaluate the model in detail.